# Reusable Template — Tabular Regression Pipeline

A dataset-agnostic notebook you can reuse for **any** "predict a continuous target from a mix of numeric/categorical/text-encoded columns" project (laptop prices, house prices, used-car prices, salaries, insurance premiums, etc.).

**How to reuse this notebook:**
1. Fill in the `CONFIG` cell (Section 1) for your dataset.
2. Fill in `clean_raw_columns()` (Section 3) with your dataset's unit-stripping / text-cleaning logic — this is the one section that's inherently dataset-specific.
3. Optionally add domain features in `engineer_domain_features()` (Section 4).
4. Run every other cell unchanged — encoding, splitting, model training, tuning, evaluation, importance, and persistence are all written generically against `CONFIG`.


## 1. Configuration

In [ ]:
CONFIG = {
    'data_path': 'laptop_price.csv',   # <-- change per project
    'target_col': 'Price',              # <-- change per project
    'id_cols': [],                       # columns to drop entirely (IDs, free-text notes, etc.)
    'cardinality_threshold': 5,          # < threshold -> one-hot, >= threshold -> target-encode
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    'log_transform_target': False,       # set True if the target is strongly right-skewed
}


## 2. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

RANDOM_STATE = CONFIG['random_state']

df_raw = pd.read_csv(CONFIG['data_path'])
df_raw = df_raw.drop(columns=[c for c in CONFIG['id_cols'] if c in df_raw.columns])
print(df_raw.shape)
df_raw.head()


## 3. Data Quality Audit (generic — reuse as-is)

In [ ]:
def audit_dataframe(df):
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(),
        'n_missing': df.isnull().sum(),
        'pct_missing': (df.isnull().mean() * 100).round(1),
        'sample': [df[c].dropna().unique()[:3].tolist() for c in df.columns],
    })
    return audit

audit = audit_dataframe(df_raw)
print("Duplicate rows:", df_raw.duplicated().sum())
audit


In [ ]:
# Scan every object column for likely disguised-missing / unit-bearing text.
for c in df_raw.select_dtypes(include='object').columns:
    vals = df_raw[c].unique()
    print(f"{c:25s} n_unique={len(vals):4d}  sample={vals[:6]}")


## 4. Cleaning & Feature Engineering (project-specific — EDIT THIS SECTION)

Replace the body of `clean_raw_columns()` with the unit-stripping / text-parsing logic your dataset needs. Common patterns are in the Cheat Sheet notebook, Section 3. Keep every transformation **idempotent** (`.astype(str)` before `.str.replace(...)`) so the cell is safe to re-run.

In [ ]:
def clean_raw_columns(df):
    """EDIT ME. Strip units / text artifacts from numeric-looking string columns.
    Example pattern (from the laptop project):

    for col in ['ram_gb', 'ssd', 'hdd']:
        df[col] = df[col].astype(str).str.replace(' GB', '').astype(int)
    """
    df = df.copy()
    # TODO: dataset-specific cleaning goes here
    return df


def engineer_domain_features(df):
    """EDIT ME (optional). Add composite / ordinal / flag features informed by
    domain knowledge. Examples: combine correlated raw columns, map a category to
    an ordinal tier, flag a premium segment, log-transform a skewed count column.
    Guard against creating a feature that's an exact linear combination of others
    you're keeping (see Background Theory, Section 2, on multicollinearity).
    """
    df = df.copy()
    # TODO: domain features go here
    return df


df = clean_raw_columns(df_raw)
df = engineer_domain_features(df)
df.head()


## 5. EDA (generic — reuse as-is, adjust column names in calls)

In [ ]:
def eda_target_distribution(df, target_col):
    print(f"{target_col} skewness: {df[target_col].skew():.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[target_col], kde=True, ax=axes[0]).set_title(f'{target_col} (raw)')
    log_col = np.log1p(df[target_col])
    sns.histplot(log_col, kde=True, ax=axes[1]).set_title(f'log1p({target_col})')
    plt.tight_layout(); plt.show()
    return log_col

df['log_target'] = eda_target_distribution(df, CONFIG['target_col'])


In [ ]:
def eda_correlation_and_vif(df, target_col, drop_cols=None):
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    numeric_df = df.select_dtypes(include='number')
    if drop_cols:
        numeric_df = numeric_df.drop(columns=[c for c in drop_cols if c in numeric_df.columns])

    plt.figure(figsize=(12, 8))
    sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation matrix'); plt.show()
    print(numeric_df.corr()[target_col].sort_values(ascending=False))

    X_vif = numeric_df.drop(columns=[target_col]).dropna()
    vif = pd.DataFrame({
        'feature': X_vif.columns,
        'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    }).sort_values('VIF', ascending=False)
    return vif

vif_table = eda_correlation_and_vif(df, CONFIG['target_col'], drop_cols=['log_target'])
vif_table


In [ ]:
def eda_outliers_iqr(df, target_col):
    Q1, Q3 = df[target_col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    mask = (df[target_col] < lower) | (df[target_col] > upper)
    print(f"Bounds: [{lower:.1f}, {upper:.1f}]  outliers: {mask.sum()} ({mask.mean():.1%})")
    return df[mask]

outliers = eda_outliers_iqr(df, CONFIG['target_col'])


## 6. Leakage-Safe Encoding (generic — reuse as-is)

In [ ]:
def split_columns_by_cardinality(df, threshold, exclude_cols):
    cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in exclude_cols]
    low_card = [c for c in cat_cols if df[c].nunique() < threshold]
    high_card = [c for c in cat_cols if df[c].nunique() >= threshold]
    return low_card, high_card


def prepare_features(df, config):
    target_col = config['target_col']
    exclude = [target_col, 'log_target']
    low_card, high_card = split_columns_by_cardinality(df, config['cardinality_threshold'], exclude)

    df_enc = pd.get_dummies(df, columns=low_card, drop_first=True)
    bool_cols = df_enc.select_dtypes(include='bool').columns
    df_enc[bool_cols] = df_enc[bool_cols].astype(int)

    y_col = 'log_target' if config['log_transform_target'] else target_col
    X = df_enc.drop(columns=[c for c in [target_col, 'log_target'] if c in df_enc.columns])
    y = df_enc[y_col] if y_col in df_enc.columns else df[y_col]

    return X, y, low_card, high_card


from sklearn.model_selection import train_test_split

X, y, low_card, high_card = prepare_features(df, CONFIG)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], random_state=RANDOM_STATE
)

target_encoding_maps = {}
global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    target_encoding_maps[col] = means
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)

print("Nulls:", X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
X_train.shape, X_test.shape


## 7. Model Zoo & Evaluation Harness (generic — reuse as-is)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    denom = np.where(y_true == 0, np.nan, y_true)
    mape = np.nanmean(np.abs((y_true - y_pred) / denom)) * 100
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE_%': mape})
    print(f"{name:22s} MAE={mae:10,.2f}  RMSE={rmse:10,.2f}  R2={r2:.3f}  MAPE={mape:5.1f}%")
    return mae, rmse, r2, mape


MODEL_ZOO = {
    'Mean baseline': None,   # handled specially below
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso': Lasso(alpha=1.0, random_state=RANDOM_STATE, max_iter=10000),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
evaluate('Mean baseline', y_test, y_pred_baseline)

fitted_models = {}
for name, model in MODEL_ZOO.items():
    if model is None:
        continue
    model.fit(X_train, y_train)
    fitted_models[name] = model
    evaluate(name, y_test, model.predict(X_test))

pd.DataFrame(results).sort_values('MAE')


## 8. Cross-Validation & Hyperparameter Tuning (generic — reuse, edit the grid)

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

candidate_name = 'Random Forest'   # <-- pick your strongest candidate from Section 7
candidate = fitted_models[candidate_name]

cv_scores = -cross_val_score(candidate, X_train, y_train, cv=CONFIG['cv_folds'],
                              scoring='neg_mean_absolute_error')
print(f"{candidate_name} {CONFIG['cv_folds']}-fold CV MAE: {cv_scores.mean():,.2f} +/- {cv_scores.std():,.2f}")

param_dist = {
    'n_estimators': [100, 200, 400, 600],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}
search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_distributions=param_dist, n_iter=30, cv=CONFIG['cv_folds'],
    scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1
)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
final_model = search.best_estimator_
evaluate('Final (tuned)', y_test, final_model.predict(X_test))


## 9. Diagnostics & Feature Importance (generic — reuse as-is)

In [ ]:
def plot_diagnostics(y_test, y_pred):
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].scatter(y_test, y_pred, alpha=0.5)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[0].plot(lims, lims, 'r--'); axes[0].set_title('Predicted vs Actual')
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.5); axes[1].axhline(0, color='r', linestyle='--')
    axes[1].set_title('Residuals vs Predicted')
    sns.histplot(residuals, kde=True, ax=axes[2]); axes[2].set_title('Residual distribution')
    plt.tight_layout(); plt.show()

y_pred_final = final_model.predict(X_test)
plot_diagnostics(y_test, y_pred_final)


In [ ]:
from sklearn.inspection import permutation_importance

def plot_feature_importance(model, X_train, X_test, y_test, top_n=10):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        idx = np.argsort(importances)[-top_n:][::-1]
        sns.barplot(x=importances[idx], y=X_train.columns[idx], ax=axes[0])
        axes[0].set_title('Impurity-based importance')

    perm = permutation_importance(model, X_test, y_test, n_repeats=10,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    pidx = perm.importances_mean.argsort()[-top_n:][::-1]
    sns.barplot(x=perm.importances_mean[pidx], y=X_test.columns[pidx], ax=axes[1])
    axes[1].set_title('Permutation importance')
    plt.tight_layout(); plt.show()

plot_feature_importance(final_model, X_train, X_test, y_test)


## 10. Persistence & Generic Inference Wrapper

In [ ]:
import joblib

def save_artifacts(model, path_prefix='model'):
    joblib.dump(model, f'{path_prefix}.pkl')
    joblib.dump({
        'target_encoding_maps': target_encoding_maps,
        'global_mean': global_mean,
        'model_columns': list(X_train.columns),
        'low_card_cols': low_card,
        'high_card_cols': high_card,
        'config': CONFIG,
    }, f'{path_prefix}_encoders.pkl')

def predict_from_raw(raw_dict, model, path_prefix='model'):
    art = joblib.load(f'{path_prefix}_encoders.pkl')
    row = pd.DataFrame([raw_dict])
    row = clean_raw_columns(row)
    row = engineer_domain_features(row)
    row = pd.get_dummies(row, columns=art['low_card_cols'], drop_first=True)
    bool_cols = row.select_dtypes(include='bool').columns
    row[bool_cols] = row[bool_cols].astype(int)
    for col in art['high_card_cols']:
        if col in row.columns:
            row[col] = row[col].map(art['target_encoding_maps'][col]).fillna(art['global_mean'])
    row = row.reindex(columns=art['model_columns'], fill_value=0)
    pred = model.predict(row)[0]
    return float(np.expm1(pred)) if art['config']['log_transform_target'] else float(pred)

save_artifacts(final_model)
print("Artifacts saved. Fill in clean_raw_columns()/engineer_domain_features() per-project, then call predict_from_raw().")


## 11. Per-Project Checklist (quick reference)

- [ ] Update `CONFIG` (data path, target column, id columns, cardinality threshold, log-transform flag)
- [ ] Implement `clean_raw_columns()` for this dataset's unit/text artifacts
- [ ] Implement `engineer_domain_features()` if domain knowledge suggests composite/ordinal/flag features
- [ ] Check the VIF table for engineered features that are linear combinations of others
- [ ] Confirm zero nulls in `X_train`/`X_test` after encoding
- [ ] Compare at least one linear and one tree-ensemble model against the mean baseline
- [ ] Cross-validate before trusting a single train/test split
- [ ] Report MAE, RMSE, R², and MAPE together — never R² alone
- [ ] Cross-check impurity-based and permutation feature importance
- [ ] Persist encoders alongside the model, not just the model
